# FullStack AI · Fondation pour l'apprentissage profond

**Objectif :** Descente de gradient, perceptron, PyTorch, convolutions et embeddings.

Lance ce notebook depuis le dossier `notebooks/`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../labs"))   # brewly, fakeapi
os.chdir("../data")                               # penguins.csv, titanic.csv, tips.csv

---
## Le neurone, les couches et les fonctions d'activation

Un réseau de neurones est une fonction construite en empilant des opérations très simples. Comprendre ces briques, c'est pouvoir lire n'importe quelle architecture, du perceptron au Transformer.

## Un neurone

Un neurone artificiel fait trois choses : il multiplie chaque entrée par un **poids**, additionne le tout avec un **biais**, puis applique une **fonction d'activation** non linéaire.

$$
a = f\Big(\sum_j w_j x_j + b\Big) = f(w \cdot x + b)
$$

*Animation interactive « neuron » : à voir sur la plateforme web.*

## Pourquoi une activation non linéaire

Sans activation, empiler des couches ne sert à rien : une suite de transformations linéaires reste une transformation linéaire. L'activation permet au réseau d'apprendre des frontières courbes.

| Activation | Formule | Usage |
| --- | --- | --- |
| **ReLU** | max(0, z) | Couches cachées : simple, rapide, le choix par défaut |
| **Sigmoïde** | 1 / (1 + e⁻ᶻ) | Sortie d'une classification binaire (une probabilité) |
| **Tanh** | (eᶻ − e⁻ᶻ) / (eᶻ + e⁻ᶻ) | Couches cachées, sortie centrée entre −1 et 1 |
| **Softmax** | eᶻⁱ / Σ eᶻʲ | Sortie d'une classification à plusieurs classes |

In [ ]:
import numpy as np
z = np.linspace(-4, 4, 9)
print("z      ", z)
print("ReLU   ", np.maximum(0, z))
print("sigmoïde", (1 / (1 + np.exp(-z))).round(3))
print("tanh   ", np.tanh(z).round(3))

## Une couche, puis un réseau

Une **couche dense** applique plusieurs neurones aux mêmes entrées : c'est un produit matriciel. Pour un lot (*batch*) de n exemples à d variables, avec une couche de k neurones :

$$
H = f(XW + b) \qquad X : (n, d) \quad W : (d, k) \quad b : (k,) \quad H : (n, k)
$$

In [ ]:
rng = np.random.default_rng(0)
X = rng.normal(size=(5, 3))                 # 5 exemples, 3 variables
W1, b1 = rng.normal(size=(3, 4)), np.zeros(4)
W2, b2 = rng.normal(size=(4, 1)), np.zeros(1)
H = np.maximum(0, X @ W1 + b1)              # couche cachée : 4 neurones, ReLU
sortie = 1 / (1 + np.exp(-(H @ W2 + b2)))   # couche de sortie : une probabilité
print(H.shape, sortie.shape)
sortie.ravel().round(3)

Ce calcul de l'entrée vers la sortie s'appelle la **propagation avant** (*forward pass*).

## La fonction de coût

Pour apprendre, il faut mesurer l'erreur. En classification, on utilise l'**entropie croisée** (*cross-entropy*) : elle punit fortement une prédiction confiante et fausse.

$$
L = -\frac{1}{n} \sum_i \big[ y_i \log p_i + (1 - y_i) \log(1 - p_i) \big]
$$

In [ ]:
y = np.array([1, 0, 1])
for p in [np.array([0.9, 0.1, 0.8]), np.array([0.6, 0.4, 0.5]), np.array([0.1, 0.9, 0.2])]:
    L = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
    print(p, "→ perte", round(L, 3))

> **À retenir** : neurone = produit scalaire + biais + activation. Couche = produit matriciel. Réseau = couches empilées. Apprendre = ajuster les poids pour réduire la fonction de coût, par descente de gradient.

### Exercice 1 · Les activations et leurs dérivées

Écris `sigmoide(z)`, `relu(z)` et leurs dérivées `d_sigmoide(z)` et `d_relu(z)` pour des tableaux NumPy. Rappel : σ'(z) = σ(z)(1 − σ(z)) ; la dérivée de ReLU vaut 1 si z > 0, 0 sinon.

<details><summary>Indice</summary>

np.maximum(0, z) pour ReLU ; (z > 0).astype(float) pour sa dérivée.
</details>

In [ ]:
import numpy as np

def sigmoide(z):
    pass

def d_sigmoide(z):
    pass

def relu(z):
    pass

def d_relu(z):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
z = np.array([-2.0, -0.5, 0.0, 0.7, 3.0])
assert np.allclose(sigmoide(z), 1 / (1 + np.exp(-z))), "sigmoide est fausse."
assert np.allclose(relu(z), [0, 0, 0, 0.7, 3]), "relu est fausse."
h = 1e-6
assert np.allclose(d_sigmoide(z), (sigmoide(z + h) - sigmoide(z - h)) / (2 * h), atol=1e-6), "d_sigmoide ne correspond pas à la dérivée numérique."
assert np.allclose(d_relu(z), [0, 0, 0, 1, 1]), "d_relu doit valoir 1 si z > 0 et 0 sinon."
print("Bravo, tout est correct.")

### Exercice 2 · Une propagation avant

Écris `forward(X, W1, b1, W2, b2)` qui calcule un réseau à une couche cachée **tanh** et une sortie **softmax**, et renvoie les probabilités (n, k). Pour la stabilité, soustrais le maximum de chaque ligne avant l'exponentielle.

<details><summary>Indice</summary>

H = np.tanh(X @ W1 + b1) ; Z = H @ W2 + b2 ; softmax ligne par ligne avec axis=1, keepdims=True.
</details>

In [ ]:
import numpy as np

def forward(X, W1, b1, W2, b2):
    pass

rng = np.random.default_rng(1)
X = rng.normal(size=(6, 4))
W1, b1 = rng.normal(size=(4, 5)), np.zeros(5)
W2, b2 = rng.normal(size=(5, 3)), np.zeros(3)

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
P = forward(X, W1, b1, W2, b2)
assert P.shape == (6, 3), f"La sortie doit avoir la forme (6, 3) ; elle a {P.shape}."
assert np.allclose(P.sum(axis=1), 1), "Chaque ligne doit sommer à 1 (softmax par ligne : axis=1, keepdims=True)."
Z = np.tanh(X @ W1 + b1) @ W2 + b2
assert np.allclose(P, np.exp(Z) / np.exp(Z).sum(axis=1, keepdims=True)), "Les probabilités ne correspondent pas : tanh en couche cachée, softmax en sortie."
assert np.isfinite(forward(X * 1000, W1, b1, W2 * 1000, b2)).all(), "Avec de très grandes valeurs, pense à soustraire le maximum de chaque ligne."
print("Bravo, tout est correct.")

### Exercice 3 · L'entropie croisée multiclasse

Écris `entropie_croisee(P, y)` : `P` contient les probabilités prédites (n, k) et `y` les indices des vraies classes (n,). Elle renvoie la moyenne de −log(probabilité de la vraie classe). Pour éviter log(0), limite les probabilités à [1e-12, 1] avec `np.clip`.

<details><summary>Indice</summary>

P[np.arange(len(y)), y] sélectionne, pour chaque ligne, la probabilité de la vraie classe.
</details>

In [ ]:
import numpy as np

def entropie_croisee(P, y):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
P = np.array([[0.7, 0.2, 0.1], [0.1, 0.8, 0.1], [0.3, 0.3, 0.4]])
y = np.array([0, 1, 2])
assert abs(entropie_croisee(P, y) - np.mean(-np.log([0.7, 0.8, 0.4]))) < 1e-9, f"entropie_croisee renvoie {entropie_croisee(P, y)}."
assert np.isfinite(entropie_croisee(np.array([[1.0, 0.0]]), np.array([1]))), "Une probabilité nulle ne doit pas donner l'infini : utilise np.clip."
print("Bravo, tout est correct.")

---
## Vecteurs et matrices

En ML, tout est vecteur ou matrice. Un exemple (un manchot, un passager) est un **vecteur** de caractéristiques. Un jeu de données est une **matrice** X : une ligne par exemple, une colonne par caractéristique.

## Vecteurs

In [ ]:
import numpy as np
u = np.array([3.0, 4.0])
v = np.array([1.0, 2.0])
print(u + v, 2 * u)
print("norme de u :", np.linalg.norm(u))     # longueur : racine(3² + 4²) = 5

## Le produit scalaire

Le produit scalaire multiplie les composantes deux à deux puis les additionne. C'est l'opération la plus fréquente en ML : une prédiction linéaire est un produit scalaire entre les caractéristiques et les poids.

$$
u \cdot v = \sum_i u_i v_i
$$

In [ ]:
print(u @ v, np.dot(u, v), (u * v).sum())   # trois écritures équivalentes : 3×1 + 4×2 = 11

La **similarité cosinus** compare la direction de deux vecteurs, entre −1 et 1. C'est elle qui mesure la proximité entre deux textes transformés en vecteurs (embeddings) dans un système RAG.

$$
\cos(u, v) = \frac{u \cdot v}{\lVert u \rVert \, \lVert v \rVert}
$$

## Matrices et produit matriciel

Le produit `A @ B` est possible si le nombre de colonnes de A égale le nombre de lignes de B. Une matrice (n, k) multipliée par une (k, m) donne une (n, m).

*Animation interactive « matmul » : à voir sur la plateforme web.*

In [ ]:
X = np.array([[1.0, 2.0],
              [3.0, 4.0],
              [5.0, 6.0]])        # 3 exemples, 2 caractéristiques
w = np.array([0.5, -1.0])         # un poids par caractéristique
b = 2.0
predictions = X @ w + b           # (3, 2) @ (2,) → (3,)
print(predictions)

Une seule ligne calcule les prédictions des 3 exemples à la fois. C'est le cœur d'un modèle linéaire et d'une couche de réseau de neurones.

In [ ]:
print(X.T)                 # transposée : lignes et colonnes échangées
print(X.T @ X)             # (2, 3) @ (3, 2) → (2, 2)
print(np.eye(2))           # matrice identité

> **À retenir** : avant chaque produit matriciel, vérifie les formes avec `.shape`. La majorité des bugs en deep learning sont des erreurs de dimensions.

### Exercice 1 · Prédire avec des poids

Un modèle prédit le prix d'un logement : `prix = 3000 × surface + 15000 × pieces + 20000`. Écris les poids dans un vecteur `w`, puis calcule en **une opération matricielle** le vecteur `prix` pour les logements de `X` (colonnes : surface, pièces).

<details><summary>Indice</summary>

w doit avoir autant de valeurs que X a de colonnes. Puis X @ w + b.
</details>

In [ ]:
import numpy as np
X = np.array([[30, 1], [55, 3], [80, 4]], dtype=float)
b = 20000

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
assert np.allclose(prix, [125000, 230000, 320000]), f"prix vaut {prix}."
print("Bravo, tout est correct.")

### Exercice 2 · Similarité cosinus

Écris `cosinus(u, v)` qui renvoie la similarité cosinus de deux vecteurs NumPy. Puis trouve `plus_proche` : le nom du document dont le vecteur est le plus similaire à `requete`.

<details><summary>Indice</summary>

u @ v divisé par le produit des normes. max(dico, key=dico.get) renvoie la clé de plus grande valeur.
</details>

In [ ]:
import numpy as np
docs = {
    "cours_python": np.array([0.9, 0.1, 0.0]),
    "recette_crepes": np.array([0.0, 0.2, 0.9]),
    "tuto_pandas": np.array([0.7, 0.6, 0.1]),
}
requete = np.array([0.8, 0.5, 0.0])

def cosinus(u, v):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
assert abs(cosinus(np.array([1.0, 0.0]), np.array([0.0, 1.0]))) < 1e-9, "Deux vecteurs perpendiculaires ont un cosinus de 0."
assert abs(cosinus(np.array([2.0, 2.0]), np.array([1.0, 1.0])) - 1) < 1e-9, "Deux vecteurs de même direction ont un cosinus de 1."
assert plus_proche == "tuto_pandas", f"plus_proche vaut {plus_proche!r}."
print("Bravo, tout est correct.")

---
## Dérivées et gradient

Entraîner un modèle, c'est **minimiser une erreur**. Pour savoir dans quelle direction modifier les paramètres, on utilise la **dérivée**.

## La dérivée : une pente

La dérivée de f en x mesure la pente de la courbe en ce point : de combien f varie quand x augmente un tout petit peu.

$$
f'(x) \approx \frac{f(x + h) - f(x)}{h} \quad \text{pour } h \text{ très petit}
$$

- Si f'(x) > 0, la fonction monte : pour descendre, il faut diminuer x.
- Si f'(x) < 0, elle descend : il faut augmenter x.
- Si f'(x) = 0, on est sur un plat, souvent un minimum ou un maximum.

In [ ]:
def f(x):
    return (x - 3) ** 2

def derivee(f, x, h=1e-6):
    return (f(x + h) - f(x - h)) / (2 * h)

for x in [0, 2, 3, 5]:
    print(f"x = {x}  pente ≈ {derivee(f, x):.3f}")

La dérivée exacte de (x − 3)² est 2(x − 3) : elle vaut −6 en 0, 0 en 3, et 4 en 5. La pente nulle en x = 3 indique le minimum.

## Les règles à connaître

| Fonction | Dérivée |
| --- | --- |
| c (constante) | 0 |
| x^n | n·x^(n−1) |
| a·x + b | a |
| e^x | e^x |
| g(h(x)) | g'(h(x)) · h'(x) (règle de la chaîne) |

La **règle de la chaîne** est au cœur de la *rétropropagation* (backpropagation), l'algorithme qui entraîne les réseaux de neurones.

## Plusieurs variables : le gradient

Quand f dépend de plusieurs paramètres, on calcule une **dérivée partielle** pour chacun, en faisant comme si les autres étaient constants. Le vecteur de toutes les dérivées partielles est le **gradient**. Il pointe dans la direction où f augmente le plus vite.

In [ ]:
import numpy as np

def g(p):
    x, y = p
    return x ** 2 + 3 * y ** 2

def gradient(f, p, h=1e-6):
    grad = np.zeros_like(p)
    for i in range(len(p)):
        e = np.zeros_like(p)
        e[i] = h
        grad[i] = (f(p + e) - f(p - e)) / (2 * h)
    return grad

print(gradient(g, np.array([1.0, 2.0])))   # exact : [2x, 6y] = [2, 12]

> **À retenir** : pour minimiser une fonction, on avance dans la direction **opposée** au gradient. C'est toute l'idée de la descente de gradient, la leçon suivante.

### Exercice 1 · Dérivée numérique

Écris `derivee(f, x, h=1e-5)` (différence centrée), puis calcule `pente`, la dérivée de `f(x) = x**3 - 2*x` en x = 2. La valeur exacte est 3·2² − 2 = 10.

<details><summary>Indice</summary>

(f(x + h) - f(x - h)) / (2 * h)
</details>

In [ ]:
def f(x):
    return x ** 3 - 2 * x

def derivee(f, x, h=1e-5):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert abs(pente - 10) < 1e-4, f"pente vaut {pente}, j'attendais 10."
assert abs(derivee(lambda x: x ** 2, 3) - 6) < 1e-4, "Ta fonction derivee doit marcher pour n'importe quelle fonction : la dérivée de x² en 3 vaut 6."
print("Bravo, tout est correct.")

### Exercice 2 · Gradient exact

Pour `perte(a, b) = (a - 1)**2 + (b + 2)**2`, écris `grad_perte(a, b)` qui renvoie le tuple des deux dérivées partielles **exactes** (pas numériques).

<details><summary>Indice</summary>

La dérivée de (a - 1)² par rapport à a est 2(a - 1) ; b est traité comme une constante.
</details>

In [ ]:
def grad_perte(a, b):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
r = grad_perte(0, 0)
assert tuple(r) == (-2, 4), f"grad_perte(0, 0) renvoie {r}, j'attendais (-2, 4)."
assert tuple(grad_perte(1, -2)) == (0, 0), "Au minimum (1, -2), le gradient doit être nul."
print("Bravo, tout est correct.")

---
## La descente de gradient

La **descente de gradient** est l'algorithme qui entraîne la quasi-totalité des modèles modernes, de la régression linéaire aux grands modèles de langage. Le principe tient en une ligne : on répète

$$
\theta \leftarrow \theta - \eta \, \nabla L(\theta)
$$

θ représente les paramètres, L la fonction de perte (l'erreur), ∇L son gradient, et η (eta) le **taux d'apprentissage** (*learning rate*) : la taille du pas.

*Animation interactive « gradient » : à voir sur la plateforme web.*

## En une dimension

In [ ]:
def L(x):
    return (x - 3) ** 2

def dL(x):
    return 2 * (x - 3)

x = 10.0
eta = 0.1
for etape in range(30):
    x = x - eta * dL(x)
print(round(x, 4))   # proche de 3, le minimum

## Le taux d'apprentissage

C'est l'hyperparamètre le plus important. Trop petit, l'apprentissage est très lent. Trop grand, on saute par-dessus le minimum et l'erreur explose.

In [ ]:
for eta in [0.01, 0.1, 0.9, 1.1]:
    x = 10.0
    for _ in range(30):
        x = x - eta * dL(x)
    print(f"eta = {eta:<5} → x = {x:.3f}")

## Une régression linéaire, à la main

On cherche la droite ŷ = w·x + b qui colle le mieux aux données. La perte est l'**erreur quadratique moyenne** (MSE) :

$$
L(w, b) = \frac{1}{n} \sum_i (w x_i + b - y_i)^2
\qquad
\frac{\partial L}{\partial w} = \frac{2}{n} \sum_i (\hat{y}_i - y_i)\, x_i
\qquad
\frac{\partial L}{\partial b} = \frac{2}{n} \sum_i (\hat{y}_i - y_i)
$$

In [ ]:
import numpy as np
import plotly.express as px

rng = np.random.default_rng(0)
x = rng.uniform(0, 10, 100)
y = 2.5 * x + 4 + rng.normal(0, 2, 100)     # vraie droite : w = 2.5, b = 4, plus du bruit

w, b, eta = 0.0, 0.0, 0.01
pertes = []
for epoque in range(500):
    y_pred = w * x + b
    erreur = y_pred - y
    pertes.append((erreur ** 2).mean())
    w -= eta * 2 * (erreur * x).mean()
    b -= eta * 2 * erreur.mean()

print(f"w = {w:.2f}, b = {b:.2f}, MSE finale = {pertes[-1]:.2f}")
px.line(y=pertes, title="La perte diminue à chaque époque",
        labels={"y": "MSE", "x": "époque"})

Le modèle retrouve presque la vraie droite (w ≈ 2,5, b ≈ 4) à partir des seules données. scikit-learn et PyTorch font exactement cela, en plus rapide et avec plus de paramètres.

> **À retenir** : une **époque** est un passage complet sur les données. Une courbe de perte qui descend puis se stabilise signifie que l'entraînement a convergé. Une courbe qui remonte signale un taux d'apprentissage trop grand.

### Exercice 1 · Minimiser une parabole

Avec une descente de gradient (eta = 0.1, 100 étapes, départ x = 0), trouve le minimum de `L(x) = (x + 4)**2 + 1`. Stocke le résultat final dans `x_min`. Écris toi-même la dérivée `dL`.

<details><summary>Indice</summary>

La dérivée de (x + 4)² + 1 est 2(x + 4). Répète x -= 0.1 * dL(x).
</details>

In [ ]:
def L(x):
    return (x + 4) ** 2 + 1

def dL(x):
    pass

x = 0.0

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert abs(x_min + 4) < 1e-3, f"x_min vaut {x_min}, le minimum est en -4."
assert dL(-4) == 0, "dL(-4) devrait valoir 0."
print("Bravo, tout est correct.")

### Exercice 2 · Régression sur les manchots

Entraîne par descente de gradient une régression `masse ≈ w × nageoire + b` sur les manchots. Les données sont **standardisées** pour que la descente converge vite (`xs`, `ys`). Utilise eta = 0.1 et 200 époques, et remplis la liste `pertes` avec la MSE à chaque époque. Compare ton `w` à la pente de `np.polyfit`.

<details><summary>Indice</summary>

Reprends la boucle de la leçon en remplaçant x et y par xs et ys, et eta par 0.1.
</details>

In [ ]:
import numpy as np
import pandas as pd
df = pd.read_csv("penguins.csv").dropna()
x = df["flipper_length_mm"].to_numpy()
y = df["body_mass_g"].to_numpy()
xs = (x - x.mean()) / x.std()
ys = (y - y.mean()) / y.std()

w, b = 0.0, 0.0
pertes = []

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
assert len(pertes) == 200, f"pertes contient {len(pertes)} valeurs, j'en attendais 200."
assert pertes[-1] < pertes[0], "La perte doit diminuer."
assert abs(w - np.polyfit(xs, ys, 1)[0]) < 1e-3, f"w vaut {w:.4f}, la pente optimale est {np.polyfit(xs, ys, 1)[0]:.4f}."
print("Bravo, tout est correct.")

---
## Le perceptron, et sa limite

Le **perceptron** (Rosenblatt, 1958) est l'ancêtre de tous les réseaux de neurones : un seul neurone, avec une activation en marche d'escalier, qui sépare deux classes par une droite (un hyperplan en dimension plus grande).

$$
\hat{y} = \begin{cases} 1 & \text{si } w \cdot x + b > 0 \\ 0 & \text{sinon} \end{cases}
$$

## La règle d'apprentissage

On parcourt les exemples ; à chaque erreur, on déplace la droite vers le bon côté :

$$
w \leftarrow w + \eta\,(y - \hat{y})\,x \qquad b \leftarrow b + \eta\,(y - \hat{y})
$$

In [ ]:
import numpy as np

def entrainer_perceptron(X, y, epoques=20, eta=1.0):
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(epoques):
        erreurs = 0
        for xi, yi in zip(X, y):
            pred = 1 if xi @ w + b > 0 else 0
            w += eta * (yi - pred) * xi
            b += eta * (yi - pred)
            erreurs += int(pred != yi)
        if erreurs == 0:
            break
    return w, b

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
for nom, y in [("ET", [0, 0, 0, 1]), ("OU", [0, 1, 1, 1]), ("OU exclusif", [0, 1, 1, 0])]:
    w, b = entrainer_perceptron(X, np.array(y))
    pred = [(1 if x @ w + b > 0 else 0) for x in X]
    print(f"{nom:12} attendu {y}  prédit {pred}  {'✓' if pred == y else '✗'}")

## La limite : le OU exclusif

Le perceptron apprend ET et OU, mais pas le **OU exclusif** (XOR) : aucune droite ne sépare (0,1) et (1,0) de (0,0) et (1,1). Ce résultat, démontré par Minsky et Papert en 1969, a longtemps freiné la recherche.

*Animation interactive « xor » : à voir sur la plateforme web.*

La solution : **ajouter une couche cachée**. Le réseau transforme d'abord l'espace, dans lequel les classes deviennent séparables par une droite. C'est le **perceptron multicouche** (MLP), entraîné par **rétropropagation** : la leçon suivante.

> **À retenir** : un seul neurone = une frontière linéaire. Les couches cachées permettent d'apprendre des frontières non linéaires : c'est toute la puissance du deep learning.

### Exercice 1 · Une classe Perceptron réutilisable

Écris une classe `Perceptron(eta=1.0, epoques=20)` avec `fit(X, y)` (règle d'apprentissage de la leçon, arrêt quand une époque se passe sans erreur, renvoie `self`, stocke `w_` et `b_` et le nombre d'erreurs de chaque époque dans `erreurs_`) et `predict(X)` qui renvoie un tableau de 0 et de 1.

<details><summary>Indice</summary>

Reprends la fonction de la leçon dans fit, avec self.w_ et self.b_. predict se fait sans boucle : (X @ w + b > 0).astype(int).
</details>

In [ ]:
import numpy as np

class Perceptron:
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
m = Perceptron()
assert m.fit(X, np.array([0, 1, 1, 1])) is m, "fit doit renvoyer self."
assert list(m.predict(X)) == [0, 1, 1, 1], "Le perceptron doit apprendre le OU."
assert m.erreurs_[-1] == 0 and len(m.erreurs_) < 20, "L'entraînement doit s'arrêter dès qu'une époque est sans erreur."
mx = Perceptron(epoques=30).fit(X, np.array([0, 1, 1, 0]))
assert len(mx.erreurs_) == 30 and min(mx.erreurs_) > 0, "Sur le OU exclusif, il reste toujours des erreurs : chaque époque doit être comptée."
rng = np.random.default_rng(0); A = rng.normal(size=(100, 2)); ya = (A[:, 0] + 2 * A[:, 1] > 0.3).astype(int)
assert (Perceptron(epoques=100).fit(A, ya).predict(A) == ya).mean() == 1, "Sur des données linéairement séparables, le perceptron doit tout classer correctement."
print("Bravo, tout est correct.")

---
## Réseau multicouche et rétropropagation

Entraîner un réseau, c'est faire une descente de gradient sur tous ses poids. Il faut donc la dérivée de la perte par rapport à chaque poids. La **rétropropagation** (*backpropagation*) calcule toutes ces dérivées efficacement, en appliquant la **règle de la chaîne** de la sortie vers l'entrée.

*Animation interactive « mlp » : à voir sur la plateforme web.*

## La règle de la chaîne, sur un neurone

Pour un neurone sigmoïde avec une perte d'entropie croisée, les dérivées se simplifient remarquablement :

$$
z = w \cdot x + b,\quad p = \sigma(z),\quad L = -[y \log p + (1-y)\log(1-p)]
\qquad\Rightarrow\qquad
\frac{\partial L}{\partial z} = p - y,\quad \frac{\partial L}{\partial w} = (p - y)\,x
$$

## Les quatre étapes d'une itération

1. **Propagation avant** : calculer la sortie en gardant les valeurs intermédiaires.
2. **Perte** : mesurer l'erreur.
3. **Rétropropagation** : calculer les gradients, couche par couche, de la fin vers le début.
4. **Mise à jour** : chaque poids recule d'un petit pas dans la direction de son gradient.

In [ ]:
import numpy as np

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([[0], [1], [1], [0]], dtype=float)          # le OU exclusif
rng = np.random.default_rng(1)
W1, b1 = rng.normal(size=(2, 8)), np.zeros(8)
W2, b2 = rng.normal(size=(8, 1)), np.zeros(1)
sig = lambda z: 1 / (1 + np.exp(-z))

for epoque in range(3001):
    # 1. propagation avant
    H = np.tanh(X @ W1 + b1)
    P = sig(H @ W2 + b2)
    # 2. perte
    L = -np.mean(y * np.log(P) + (1 - y) * np.log(1 - P))
    # 3. rétropropagation
    dZ2 = (P - y) / len(X)               # dL/dz de la sortie
    dW2, db2 = H.T @ dZ2, dZ2.sum(axis=0)
    dH = dZ2 @ W2.T                      # on remonte dans la couche cachée
    dZ1 = dH * (1 - H ** 2)              # dérivée de tanh
    dW1, db1 = X.T @ dZ1, dZ1.sum(axis=0)
    # 4. mise à jour
    for param, grad in [(W1, dW1), (b1, db1), (W2, dW2), (b2, db2)]:
        param -= 0.5 * grad
    if epoque % 1000 == 0:
        print(f"époque {epoque:4}  perte {L:.4f}")
print("prédictions :", P.ravel().round(3))

La couche cachée a appris à transformer l'espace : le OU exclusif, impossible pour le perceptron, est résolu.

## Vérifier ses gradients

Un bug dans la rétropropagation ne fait pas planter le programme : le réseau apprend juste mal. On vérifie donc les gradients analytiques en les comparant à des **dérivées numériques** (différences finies), comme en semaine de maths.

In [ ]:
def perte(W1_):
    H = np.tanh(X @ W1_ + b1)
    P = sig(H @ W2 + b2)
    return -np.mean(y * np.log(P) + (1 - y) * np.log(1 - P))

h, i, j = 1e-6, 0, 3
W_plus, W_moins = W1.copy(), W1.copy()
W_plus[i, j] += h; W_moins[i, j] -= h
H = np.tanh(X @ W1 + b1); P = sig(H @ W2 + b2)
dZ1 = ((P - y) / len(X)) @ W2.T * (1 - H ** 2)
print("numérique :", (perte(W_plus) - perte(W_moins)) / (2 * h), " analytique :", (X.T @ dZ1)[i, j])

PyTorch fait tout ce calcul automatiquement (*autograd*) : c'est la leçon suivante. Mais l'avoir codé une fois toi-même, c'est ce qui te permettra de comprendre les problèmes de gradients qui disparaissent ou explosent.

> **À retenir** : la rétropropagation = la règle de la chaîne appliquée couche par couche, de la sortie vers l'entrée. Les formes des gradients sont toujours celles des paramètres : vérifie-les avec `.shape`.

### Exercice 1 · Les gradients d'un neurone

Pour un neurone sigmoïde avec perte d'entropie croisée moyenne, écris `gradients(X, y, w, b)` qui renvoie `(dw, db)` en utilisant la formule de la leçon : dL/dz = (p − y) / n.

<details><summary>Indice</summary>

p = sigmoïde(X @ w + b) ; dz = (p − y) / n ; dw = X.T @ dz ; db = dz.sum().
</details>

In [ ]:
import numpy as np

def gradients(X, y, w, b):
    pass

rng = np.random.default_rng(0)
X = rng.normal(size=(20, 3))
y = (X[:, 0] > 0).astype(float)
w, b = rng.normal(size=3), 0.1

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
def L(w_, b_):
    p = 1 / (1 + np.exp(-(X @ w_ + b_)))
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
dw, db = gradients(X, y, w, b)
h = 1e-6
num = np.array([(L(w + h * e, b) - L(w - h * e, b)) / (2 * h) for e in np.eye(3)])
assert np.shape(dw) == (3,), f"dw doit avoir la forme de w, (3,) ; il a {np.shape(dw)}."
assert np.allclose(dw, num, atol=1e-6), f"dw = {np.round(dw, 5)} ne correspond pas au gradient numérique {np.round(num, 5)}."
assert abs(db - (L(w, b + h) - L(w, b - h)) / (2 * h)) < 1e-6, "db ne correspond pas au gradient numérique."
print("Bravo, tout est correct.")

### Exercice 2 · Un MLP qui apprend le OU exclusif

Écris `entrainer_xor(epoques=3000, eta=0.5, graine=1)` qui entraîne le réseau de la leçon (8 neurones cachés tanh, sortie sigmoïde) sur XOR et renvoie `(P, pertes)` : les prédictions finales (4, 1) et la liste des pertes à chaque époque.

<details><summary>Indice</summary>

Copie la boucle de la leçon dans la fonction. Attention à l'ordre : calcule dZ1 avec W2 avant de mettre W2 à jour.
</details>

In [ ]:
import numpy as np
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([[0], [1], [1], [0]], dtype=float)

def entrainer_xor(epoques=3000, eta=0.5, graine=1):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
P, pertes = entrainer_xor()
assert np.shape(P) == (4, 1), f"P doit avoir la forme (4, 1) ; il a {np.shape(P)}."
assert ((np.asarray(P) > 0.5) == y).all(), f"Le réseau n'a pas appris le OU exclusif : {np.asarray(P).ravel().round(3)}."
assert len(pertes) == 3000 and pertes[-1] < 0.1 * pertes[0], "La perte doit être enregistrée à chaque époque et diminuer fortement."
print("Bravo, tout est correct.")

---
## Lire et écrire du PyTorch

**PyTorch** est la bibliothèque de deep learning utilisée au FullStack. Elle fait trois choses que tu viens de faire à la main : calculer sur des **tenseurs** (des tableaux NumPy qui tournent aussi sur GPU), calculer les gradients automatiquement (**autograd**), et fournir des briques prêtes (couches, pertes, optimiseurs). PyTorch ne tourne pas dans le navigateur : les blocs ci-dessous sont à exécuter sur ton ordinateur ou dans Google Colab.

## Les tenseurs et l'autograd

```python
import torch

x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
w = torch.randn(2, 1, requires_grad=True)    # PyTorch suivra les opérations sur w
perte = ((x @ w) ** 2).mean()
perte.backward()                             # rétropropagation automatique
print(w.grad)                                # dPerte/dw, de la forme de w
```

## La correspondance avec ce que tu sais faire

| Ce que tu as codé en NumPy | En PyTorch |
| --- | --- |
| `W, b` et `X @ W + b` | `nn.Linear(d, k)` |
| `np.tanh`, `np.maximum(0, z)` | `nn.Tanh()`, `nn.ReLU()` |
| Softmax + entropie croisée | `nn.CrossEntropyLoss()` (prend les scores bruts) |
| Gradients à la main | `perte.backward()` |
| `W -= eta * dW` | `optimizer.step()` (SGD, Adam…) |
| Boucle sur des lots | `DataLoader` |

## Une boucle d'entraînement complète

C'est **toujours** la même structure : tu la retrouveras dans tous les projets.

```python
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

modele = nn.Sequential(
    nn.Linear(4, 16), nn.ReLU(),
    nn.Linear(16, 3),                 # 3 classes : scores bruts (logits)
)
perte_fn = nn.CrossEntropyLoss()
optimiseur = torch.optim.Adam(modele.parameters(), lr=1e-2)
loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

for epoque in range(50):
    modele.train()
    for xb, yb in loader:
        optimiseur.zero_grad()            # 1. remettre les gradients à zéro
        logits = modele(xb)               # 2. propagation avant
        perte = perte_fn(logits, yb)      # 3. perte
        perte.backward()                  # 4. rétropropagation
        optimiseur.step()                 # 5. mise à jour
    modele.eval()
    with torch.no_grad():                 # pas de gradients pendant l'évaluation
        acc = (modele(X_test).argmax(dim=1) == y_test).float().mean()
    print(epoque, round(perte.item(), 4), round(acc.item(), 3))
```

Les pièges classiques : oublier `zero_grad()` (les gradients s'additionnent d'un lot à l'autre), appliquer une softmax avant `CrossEntropyLoss` (elle l'applique déjà), oublier `model.eval()` et `torch.no_grad()` pendant l'évaluation.

## Écrire son propre module

```python
class MonReseau(nn.Module):
    def __init__(self, d, cache, k):
        super().__init__()
        self.couche1 = nn.Linear(d, cache)
        self.couche2 = nn.Linear(cache, k)

    def forward(self, x):
        return self.couche2(torch.relu(self.couche1(x)))
```

Tu reconnais la POO de la Fondation : héritage de `nn.Module`, `super().__init__()`, et une méthode `forward`.

## Les mêmes briques, en NumPy

Pour bien comprendre ce que fait `nn.Linear`, on l'écrit avec sa méthode `backward` : c'est l'exercice de cette leçon.

In [ ]:
import numpy as np

class Lineaire:
    """Comme nn.Linear : y = x W + b, avec la rétropropagation."""
    def __init__(self, d, k, graine=0):
        rng = np.random.default_rng(graine)
        self.W = rng.normal(0, np.sqrt(2 / d), size=(d, k))    # initialisation de He
        self.b = np.zeros(k)

    def forward(self, x):
        self.x = x                       # gardé pour la rétropropagation
        return x @ self.W + self.b

couche = Lineaire(4, 3)
print(couche.forward(np.ones((2, 4))).shape)

> **À retenir** : zero_grad, forward, perte, backward, step. Cinq lignes au cœur de tout entraînement PyTorch.

**À faire :**

- [ ] Installer PyTorch (pip install torch) ou ouvrir un notebook Google Colab
- [ ] Recopier la boucle d'entraînement et l'appliquer aux manchots (4 variables, 3 espèces)
- [ ] Comparer SGD et Adam : quelle courbe de perte descend le plus vite ?
- [ ] Lire le tutoriel officiel « Learn the Basics » de PyTorch, en anglais

### Exercice 1 · Une couche linéaire avec rétropropagation

Complète la classe `Lineaire` avec `backward(grad_sortie)` : elle reçoit dL/dy de forme (n, k), stocke `self.dW` (forme (d, k)) et `self.db` (forme (k,)), et renvoie dL/dx (forme (n, d)). Rappel : y = xW + b.

<details><summary>Indice</summary>

dW = x.T @ grad_sortie ; db = somme sur l'axe 0 ; dx = grad_sortie @ W.T. Vérifie les formes.
</details>

In [ ]:
import numpy as np

class Lineaire:
    def __init__(self, d, k, graine=0):
        rng = np.random.default_rng(graine)
        self.W = rng.normal(0, np.sqrt(2 / d), size=(d, k))
        self.b = np.zeros(k)

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b

    def backward(self, grad_sortie):
        pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
rng = np.random.default_rng(3)
c = Lineaire(4, 3, graine=1)
x = rng.normal(size=(5, 4)); cible = rng.normal(size=(5, 3))
perte = lambda: 0.5 * ((c.forward(x) - cible) ** 2).sum()
yv = c.forward(x)
dx = c.backward(yv - cible)
assert dx.shape == (5, 4) and c.dW.shape == (4, 3) and c.db.shape == (3,), "Les formes doivent être dx (n, d), dW (d, k), db (k,)."
h = 1e-6; W0 = c.W.copy()
c.W = W0.copy(); c.W[2, 1] += h; lp = perte(); c.W = W0.copy(); c.W[2, 1] -= h; lm = perte(); c.W = W0
assert abs(c.dW[2, 1] - (lp - lm) / (2 * h)) < 1e-4, "dW ne correspond pas au gradient numérique : dW = x.T @ grad_sortie."
x2 = x.copy(); x2[1, 3] += h; lp = 0.5 * ((x2 @ c.W + c.b - cible) ** 2).sum(); x2[1, 3] -= 2 * h; lm = 0.5 * ((x2 @ c.W + c.b - cible) ** 2).sum()
assert abs(dx[1, 3] - (lp - lm) / (2 * h)) < 1e-4, "dx ne correspond pas : dx = grad_sortie @ W.T."
print("Bravo, tout est correct.")

---
## Les convolutions et les CNN

Une image de 224 × 224 pixels en couleur contient 150 528 valeurs. Une couche dense qui relie chaque pixel à 1 000 neurones aurait 150 millions de poids, et ignorerait que deux pixels voisins sont liés. Les **réseaux convolutifs** (CNN) résolvent ce problème avec une idée simple : faire glisser un petit **filtre** sur toute l'image.

*Animation interactive « conv » : à voir sur la plateforme web.*

## L'opération de convolution

Un filtre (ou *noyau*, *kernel*) est une petite matrice, par exemple 3 × 3. On le place sur chaque zone de l'image, on multiplie terme à terme et on additionne : cela donne un pixel de la **carte de caractéristiques** (*feature map*).

In [ ]:
import numpy as np

def convolution(image, filtre):
    h, w = image.shape
    k = filtre.shape[0]
    sortie = np.zeros((h - k + 1, w - k + 1))
    for i in range(sortie.shape[0]):
        for j in range(sortie.shape[1]):
            sortie[i, j] = (image[i:i + k, j:j + k] * filtre).sum()
    return sortie

image = np.zeros((6, 6)); image[:, 3:] = 1          # moitié gauche noire, moitié droite blanche
bord_vertical = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]])
convolution(image, bord_vertical)

Le filtre réagit fort là où l'intensité change de gauche à droite : il **détecte les bords verticaux**. Dans un CNN, les filtres ne sont pas choisis à la main : ils sont **appris**. Les premières couches apprennent des bords et des textures, les suivantes des formes, puis des objets.

## Les réglages d'une convolution

- **Taille du filtre** : 3 × 3 le plus souvent.
- **Pas** (*stride*) : de combien on décale le filtre. Un pas de 2 divise la taille par deux.
- **Remplissage** (*padding*) : des zéros autour de l'image pour garder la même taille en sortie.
- **Nombre de filtres** : chaque filtre produit une carte ; 32 filtres donnent 32 canaux.

Taille de sortie, pour une entrée de taille n, un filtre k, un remplissage p et un pas s :

$$
\text{sortie} = \left\lfloor \frac{n + 2p - k}{s} \right\rfloor + 1
$$

## Le pooling

Le **max pooling** garde le maximum de chaque bloc (souvent 2 × 2) : il réduit la taille et rend le réseau tolérant aux petits décalages.

## L'architecture typique

```python
from torch import nn

cnn = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 224 → 112
    nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 112 → 56
    nn.Flatten(),
    nn.Linear(64 * 56 * 56, 2),       # chat ou chien
)
```

En pratique, on part presque toujours d'un réseau **pré-entraîné** sur des millions d'images (ResNet, EfficientNet) et on n'entraîne que les dernières couches : c'est l'**apprentissage par transfert**, au programme du module Adaptation et évaluation.

> **À retenir** : une convolution = un petit filtre partagé qui glisse sur l'image. Peu de paramètres, et le réseau détecte un motif où qu'il se trouve.

### Exercice 1 · Convolution avec un pas

Écris `conv2d(image, filtre, pas=1)` : une convolution sans remplissage (*valid*), avec un pas réglable. La taille de sortie est ⌊(n − k) / pas⌋ + 1 dans chaque dimension.

<details><summary>Indice</summary>

La zone commence en (i × pas, j × pas). La taille de sortie se calcule avec // (division entière).
</details>

In [ ]:
import numpy as np

def conv2d(image, filtre, pas=1):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
from scipy.signal import correlate2d
rng = np.random.default_rng(0)
img = rng.normal(size=(7, 9)); f = rng.normal(size=(3, 3))
assert np.allclose(conv2d(img, f), correlate2d(img, f, mode="valid")), "Avec un pas de 1, le résultat doit correspondre à une corrélation 2D « valid »."
s2 = conv2d(img, f, pas=2)
assert s2.shape == (3, 4), f"Avec un pas de 2 sur une image 7 × 9, la sortie doit être 3 × 4 ; elle est {s2.shape}."
assert np.allclose(s2, correlate2d(img, f, mode="valid")[::2, ::2]), "Les valeurs avec un pas de 2 sont fausses."
print("Bravo, tout est correct.")

### Exercice 2 · Max pooling

Écris `max_pool(carte, taille=2)` qui découpe la carte en blocs `taille × taille` sans chevauchement et garde le maximum de chacun. Si la taille ne divise pas exactement la carte, ignore les lignes et colonnes en trop.

<details><summary>Indice</summary>

Astuce NumPy : reshape(h, taille, w, taille) puis .max(axis=(1, 3)). Une double boucle marche aussi.
</details>

In [ ]:
import numpy as np

def max_pool(carte, taille=2):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
assert np.array_equal(max_pool(np.arange(16).reshape(4, 4)), [[5, 7], [13, 15]]), f"Sur une carte 4 × 4, j'attends [[5, 7], [13, 15]] ; tu renvoies {max_pool(np.arange(16).reshape(4, 4))}."
m = max_pool(np.arange(35).reshape(5, 7))
assert m.shape == (2, 3) and m[1, 2] == 26, f"Sur une carte 5 × 7, la sortie doit être 2 × 3 (le reste est ignoré) ; tu as {m.shape}."
print("Bravo, tout est correct.")

---
## Les embeddings

Un réseau de neurones ne manipule que des nombres. Comment lui donner des mots, des produits ou des utilisateurs ? Avec des **embeddings** (plongements) : chaque élément est représenté par un **vecteur dense**, appris de façon à ce que des éléments semblables aient des vecteurs proches. C'est la brique de base des LLM, des moteurs de recherche sémantique et des systèmes de recommandation.

## One-hot contre embedding

Le **one-hot** représente chaque mot d'un vocabulaire de 50 000 mots par un vecteur de 50 000 zéros et un seul 1 : énorme, et tous les mots sont à la même distance les uns des autres. Un **embedding** le représente par un vecteur de quelques centaines de valeurs, et la géométrie a du sens.

En pratique, une couche d'embedding est simplement une **table** : une matrice (taille du vocabulaire × dimension) dont on lit la ligne correspondant au mot. Ses valeurs sont des poids appris comme les autres.

In [ ]:
import numpy as np
vocab = {"chat": 0, "chien": 1, "voiture": 2}
rng = np.random.default_rng(0)
table = rng.normal(size=(len(vocab), 4))          # nn.Embedding(3, 4) en PyTorch
table[vocab["chien"]]

## La géométrie du sens

Des embeddings entraînés sur beaucoup de texte (word2vec, GloVe, puis ceux des Transformers) capturent des relations : les directions de l'espace encodent des notions comme le genre, le temps ou le pays.

*Animation interactive « embed » : à voir sur la plateforme web.*

Voici des vecteurs simplifiés à 4 dimensions (royauté, masculin, humain, animal), pour comprendre le principe :

In [ ]:
mots = {
    "roi":   np.array([0.9, 0.8, 0.9, 0.0]),
    "reine": np.array([0.9, -0.8, 0.9, 0.0]),
    "homme": np.array([0.1, 0.8, 0.9, 0.0]),
    "femme": np.array([0.1, -0.8, 0.9, 0.0]),
    "chat":  np.array([0.0, 0.1, 0.1, 0.9]),
    "lion":  np.array([0.6, 0.3, 0.0, 0.9]),
}
cos = lambda a, b: a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
cible = mots["roi"] - mots["homme"] + mots["femme"]
sorted(((round(cos(cible, v), 3), m) for m, v in mots.items()), reverse=True)[:3]

« roi − homme + femme ≈ reine » : la célèbre analogie de word2vec.

## Comment on les apprend

- **word2vec** : on entraîne un petit réseau à prédire un mot à partir de ses voisins ; les poids appris sont les embeddings.
- **Dans un modèle** : la table d'embeddings est la première couche, entraînée avec tout le reste pour la tâche visée.
- **Modèles d'embeddings dédiés** : on obtient directement un vecteur pour une phrase ou un document, via une API ou un modèle Hugging Face. C'est ce qu'utilise un RAG.

## Classer des textes avec des embeddings

La méthode la plus simple : faire la **moyenne** des vecteurs des mots d'un texte, puis donner ce vecteur à un classifieur.

In [ ]:
from sklearn.linear_model import LogisticRegression

vecteurs = {"super": [1, 0.2], "génial": [0.9, 0.1], "top": [0.8, 0.3], "nul": [-1, 0.1], "horrible": [-0.9, 0.2],
            "décevant": [-0.8, 0.0], "film": [0, 1], "acteurs": [0.05, 0.9]}
def vectoriser(texte):
    vs = [vecteurs[m] for m in texte.split() if m in vecteurs]
    return np.mean(vs, axis=0) if vs else np.zeros(2)

textes = ["film super", "acteurs génial", "film top", "film nul", "acteurs horrible", "film décevant"]
y = [1, 1, 1, 0, 0, 0]
clf = LogisticRegression().fit([vectoriser(t) for t in textes], y)
clf.predict([vectoriser("acteurs top"), vectoriser("film horrible")])

> **À retenir** : un embedding est un vecteur appris qui place les éléments semblables près les uns des autres. On les compare par similarité cosinus. C'est le pont entre le deep learning, la recherche sémantique et les LLM.

### Exercice 1 · Les plus proches voisins

Écris `voisins(mot, mots, k=2)` qui renvoie la liste des `k` mots les plus similaires à `mot` (similarité cosinus), **sans** le mot lui-même, du plus au moins similaire.

<details><summary>Indice</summary>

Calcule la similarité cosinus avec chaque autre mot, trie par ordre décroissant, garde les k premiers.
</details>

In [ ]:
import numpy as np

mots = {
    "roi": np.array([0.9, 0.8, 0.9, 0.0]), "reine": np.array([0.9, -0.8, 0.9, 0.0]),
    "homme": np.array([0.1, 0.8, 0.9, 0.0]), "femme": np.array([0.1, -0.8, 0.9, 0.0]),
    "chat": np.array([0.0, 0.1, 0.1, 0.9]), "lion": np.array([0.6, 0.3, 0.0, 0.9]),
}

def voisins(mot, mots, k=2):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert voisins("chat", mots, 1) == ["lion"], f"Le voisin de chat devrait être lion ; tu renvoies {voisins('chat', mots, 1)}."
r = voisins("roi", mots, 3)
assert "roi" not in r and len(r) == 3, "Le mot lui-même ne doit pas faire partie de ses voisins."
assert r[0] == "homme", f"Le plus proche de roi devrait être homme ; tu renvoies {r}."
print("Bravo, tout est correct.")

### Exercice 2 · Résoudre une analogie

Écris `analogie(a, b, c, mots)` qui résout « a est à b ce que c est à ? » : elle calcule `mots[b] - mots[a] + mots[c]` et renvoie le mot le plus similaire (cosinus), en **excluant** a, b et c. Par exemple `analogie("homme", "roi", "femme", mots)` doit renvoyer `"reine"`.

<details><summary>Indice</summary>

cible = mots[b] − mots[a] + mots[c], puis max(candidats, key=similarité avec cible).
</details>

In [ ]:
import numpy as np

mots = {
    "roi": np.array([0.9, 0.8, 0.9, 0.0]), "reine": np.array([0.9, -0.8, 0.9, 0.0]),
    "homme": np.array([0.1, 0.8, 0.9, 0.0]), "femme": np.array([0.1, -0.8, 0.9, 0.0]),
    "prince": np.array([0.7, 0.8, 0.9, 0.1]), "princesse": np.array([0.7, -0.8, 0.9, 0.1]),
    "chat": np.array([0.0, 0.1, 0.1, 0.9]), "lion": np.array([0.6, 0.3, 0.0, 0.9]),
}

def analogie(a, b, c, mots):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert analogie("homme", "roi", "femme", mots) == "reine", f"homme → roi, femme → {analogie('homme', 'roi', 'femme', mots)} (attendu : reine)."
assert analogie("roi", "prince", "reine", mots) == "princesse", "roi → prince, reine → princesse."
assert analogie("femme", "reine", "homme", mots) == "roi", "Pense à exclure a, b et c des candidats."
print("Bravo, tout est correct.")

---
## Projet 3 · Ta régression linéaire de A à Z

Objectif : écrire une **classe** de régression linéaire entraînée par descente de gradient, avec l'interface `fit` / `predict` de scikit-learn. Tu réutilises les classes (semaine 1), NumPy (semaine 2) et les maths de cette semaine.

Ta classe doit gérer **plusieurs** caractéristiques : X est une matrice (n, k), les poids `coef_` un vecteur de k valeurs, et le biais `intercept_` un nombre. Les gradients en version matricielle :

$$
\hat{y} = X w + b \qquad
\nabla_w L = \frac{2}{n} X^\top (\hat{y} - y) \qquad
\frac{\partial L}{\partial b} = \frac{2}{n} \sum_i (\hat{y}_i - y_i)
$$

**À faire :**

- [ ] Copier ta classe dans un fichier regression_gd.py
- [ ] Tracer la courbe de perte dans un notebook
- [ ] Expliquer dans le README ce que représentent coef_ et intercept_
- [ ] Publier sur GitHub

### Exercice 1 · La classe RegressionGD

Complète `RegressionGD`. `fit(X, y)` initialise `coef_` à des zéros et `intercept_` à 0, puis fait `n_epoques` étapes de descente de gradient en remplissant `self.pertes_` (MSE à chaque époque) ; elle renvoie `self`. `predict(X)` renvoie `X @ coef_ + intercept_`.

<details><summary>Indice</summary>

n, k = X.shape. L'erreur est X @ coef_ + intercept_ - y ; le gradient des poids est 2 / n * X.T @ erreur.
</details>

In [ ]:
import numpy as np

class RegressionGD:
    def __init__(self, eta=0.1, n_epoques=500):
        self.eta = eta
        self.n_epoques = n_epoques

    def fit(self, X, y):
        pass

    def predict(self, X):
        pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
rng = np.random.default_rng(5)
X_t = rng.normal(size=(300, 3))
y_t = X_t @ np.array([1.5, -2.0, 0.7]) + 4 + rng.normal(0, 0.05, 300)
m_t = RegressionGD(eta=0.1, n_epoques=500)
assert m_t.fit(X_t, y_t) is m_t, "fit doit renvoyer self."
assert np.allclose(m_t.coef_, [1.5, -2.0, 0.7], atol=0.05), f"coef_ vaut {m_t.coef_}, j'attendais environ [1.5, -2.0, 0.7]."
assert abs(m_t.intercept_ - 4) < 0.05, f"intercept_ vaut {m_t.intercept_}, j'attendais environ 4."
assert len(m_t.pertes_) == 500 and m_t.pertes_[-1] < m_t.pertes_[0], "pertes_ doit contenir une MSE par époque, en baisse."
assert m_t.predict(X_t[:2]).shape == (2,), "predict doit renvoyer une prédiction par ligne."
print("Bravo, tout est correct.")

### Exercice 2 · Appliquer aux manchots

Utilise ta classe (recopie-la au début) pour prédire `body_mass_g` à partir de `flipper_length_mm`, `bill_length_mm` et `bill_depth_mm`, standardisés. Calcule `rmse`, la racine de l'erreur quadratique moyenne **en grammes** sur toutes les données.

<details><summary>Indice</summary>

rmse = np.sqrt(((predictions - y) ** 2).mean()). Avec des données standardisées, eta = 0.1 et 1 000 époques suffisent.
</details>

In [ ]:
import numpy as np
import pandas as pd
# recopie ta classe RegressionGD ici

df = pd.read_csv("penguins.csv").dropna()
cols = ["flipper_length_mm", "bill_length_mm", "bill_depth_mm"]
X = df[cols].to_numpy()
X = (X - X.mean(axis=0)) / X.std(axis=0)
y = df["body_mass_g"].to_numpy()

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert 380 < rmse < 400, f"rmse vaut {rmse:.1f} g ; avec ces 3 variables on obtient environ 390 g. Ton modèle a-t-il convergé ?"
print("Bravo, tout est correct.")

---
## Projet · Ton réseau de neurones from scratch

L'équivalent des projets « Créez une classe de perceptron réutilisable » et « Réseaux neuronaux avec PyTorch » : un réseau de neurones complet, écrit en NumPy, qui classe les espèces de manchots. Tu le réécriras ensuite en PyTorch sur ton ordinateur, et tu compareras.

Architecture : 4 variables standardisées → couche cachée de 16 neurones ReLU → 3 sorties softmax, perte d'entropie croisée, descente de gradient sur tout le jeu d'entraînement.

Les gradients, pour une sortie softmax avec entropie croisée et des étiquettes en one-hot Y :

$$
dZ_2 = \frac{P - Y}{n} \qquad dW_2 = H^\top dZ_2 \qquad dH = dZ_2 W_2^\top \qquad dZ_1 = dH \odot \mathbb{1}[Z_1 > 0] \qquad dW_1 = X^\top dZ_1
$$

**À faire :**

- [ ] Réécrire le même réseau en PyTorch (nn.Sequential, CrossEntropyLoss, Adam) dans un notebook
- [ ] Tracer les courbes de perte NumPy et PyTorch sur le même graphique
- [ ] Entraîner un petit CNN sur MNIST ou chats contre chiens dans Google Colab avec un GPU
- [ ] Publier le tout sur GitHub avec un README qui explique la rétropropagation en 5 lignes

### Exercice 1 · La classe MLP

Écris une classe `MLP(d, cache, k, eta=0.1, epoques=300, graine=0)`. Le constructeur initialise `W1` (d, cache), `W2` (cache, k) avec une loi normale de He (écart-type √(2 / entrées)) et des biais nuls. `fit(X, y)` (y : indices de classe) fait `epoques` itérations de descente de gradient en stockant la perte de chaque époque dans `self.pertes_`, et renvoie `self`. `predict_proba(X)` et `predict(X)` complètent l'interface.

<details><summary>Indice</summary>

Sépare une méthode _forward qui renvoie Z1, H et P. Dans fit, convertis y en one-hot avec np.eye(k)[y].
</details>

In [ ]:
import numpy as np

class MLP:
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
m = MLP(4, 16, 3)
assert m.W1.shape == (4, 16) and m.W2.shape == (16, 3) and m.b1.shape == (16,), "Vérifie les formes des poids."
rng = np.random.default_rng(0)
X = rng.normal(size=(200, 2)); y = (X[:, 0] * X[:, 1] > 0).astype(int)
m2 = MLP(2, 16, 2, eta=0.5, epoques=600)
assert m2.fit(X, y) is m2, "fit doit renvoyer self."
assert len(m2.pertes_) == 600 and m2.pertes_[-1] < m2.pertes_[0] * 0.5, "La perte doit être enregistrée à chaque époque et diminuer."
P = m2.predict_proba(X)
assert P.shape == (200, 2) and np.allclose(P.sum(axis=1), 1), "predict_proba doit renvoyer des probabilités qui somment à 1."
assert (m2.predict(X) == y).mean() > 0.9, f"Sur ce problème non linéaire (XOR en 2D), l'exactitude devrait dépasser 90 % ; tu as {(m2.predict(X) == y).mean():.2f}."
print("Bravo, tout est correct.")

### Exercice 2 · Classer les manchots

Utilise ta classe `MLP` (recopie-la) sur les manchots : standardise les 4 mesures **avec la moyenne et l'écart-type du jeu d'entraînement**, entraîne `MLP(4, 16, 3, eta=0.1, epoques=300)` sur l'entraînement et calcule `acc_test`, l'exactitude sur le test.

<details><summary>Indice</summary>

moy et ecart se calculent sur X[train] seulement (pas de fuite), puis on transforme tout X avec eux.
</details>

In [ ]:
import numpy as np
import pandas as pd

# recopie ta classe MLP ici

p = pd.read_csv("penguins.csv").dropna()
cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
classes = sorted(p["species"].unique())
X = p[cols].to_numpy()
y = p["species"].map({c: i for i, c in enumerate(classes)}).to_numpy()
idx = np.random.default_rng(42).permutation(len(X))
train, test = idx[:266], idx[266:]

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert acc_test >= 0.95, f"acc_test vaut {acc_test:.3f} ; avec des données standardisées, on dépasse 95 %."
assert 0 <= acc_test <= 1, "acc_test doit être une proportion entre 0 et 1."
print("Bravo, tout est correct.")